# Yelp Pipeline - 01: Ingestion, Chunking e Preprocessing Dati su Larga Scala (~5 GB)
### Ottimizzato con Supporto Nativo per Google Colab Premium / High-RAM Runtime

Questo notebook implementa la **prima fase fondamentale** della pipeline analitica su Yelp:
1. **Supporto Flessibile per Dataset da 5 GB**:
   - **Modalità Google Colab Premium / High-RAM (~52 GB RAM)**: Sfrutta la memoria estesa e l'accelerazione multi-thread del motore **PyArrow** (e opzionalmente **cuDF GPU**) per caricare ed elaborare direttamente in memoria o a blocchi giganti (*mega-chunk da 1.000.000 di righe*) i dati grezzi in tempi record, senza forzare chunking restrittivi.
   - **Modalità Standard**: Mantiene comunque il supporto di fallback a streaming a blocchi (*chunksize = 100.000*) per runtime con limiti di memoria standard (12 GB RAM).
2. **Filtraggio Selettivo di Dominio**: Isolamento delle attività del settore della ristorazione (`Restaurants` / `Food`), con stato operativo attivo (`is_open = 1`) e localizzazione nelle principali aree metropolitane (es. Toronto, Las Vegas, Phoenix).
3. **Data Cleaning e Schema Normalization**: Standardizzazione degli schemi, parsing dei tipi di dato (`datetime`, categorici, interi a basso impatto di memoria), gestione valori mancanti.
4. **Esportazione in Apache Parquet su Google Drive**: Serializzazione in formato colonnare compresso (`.parquet`) che riduce la dimensione dei dati da oltre 5 GB a circa 50-100 MB, consentendo caricamenti istantanei (1-2 secondi) nei successivi notebook.

## Configurazione Ambiente e Dipendenze

In [1]:
# Installazione dipendenze essenziali per gestione Parquet, multi-threading e logging avanzato
%pip install -q pyarrow fastparquet loguru tabulate psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.9 MB/s eta 0:00:00


## Accelerazione Hardware (Opzionale cuDF su GPU Colab Premium)

In [2]:
# Tentativo di abilitazione cuDF pandas per esecuzione GPU-accelerata su Colab Premium (es. L4 / A100 / T4)
try:
    %load_ext cudf.pandas
    print("Accelerazione GPU cuDF abilitata con successo!")
except Exception:
    print("Esecuzione su CPU standard ad alte prestazioni (PyArrow multi-threaded).")

Accelerazione GPU cuDF abilitata con successo!


## Montaggio Google Drive e Configurazione Hardware Colab

In [3]:
import os
import sys
import json
import time
import psutil
from pathlib import Path
import numpy as np
import pandas as pd
from loguru import logger

# Configurazione fuso orario europeo
os.environ['TZ'] = 'Europe/Rome'
time.tzset()

# Configurazione logger avanzato loguru
logger.remove()
logger.add(
    sys.stdout,
    colorize=True,
    format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{message}</cyan>"
)

# Connessione a Google Drive con rilevamento intelligente cartelle
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive')

    # Rilevamento cartella dati grezzi (priorità a /content/drive/MyDrive/data visibile nello screenshot)
    if (drive_root / 'data').exists():
        RAW_DIR = drive_root / 'data'
    elif (drive_root / 'yelp_raw').exists():
        RAW_DIR = drive_root / 'yelp_raw'
    elif (drive_root / 'yelp_data').exists():
        RAW_DIR = drive_root / 'yelp_data'
    else:
        RAW_DIR = drive_root / 'data'

    DRIVE_DIR = drive_root / 'yelp_data'
except Exception:
    DRIVE_DIR = Path('./yelp_data')
    RAW_DIR = Path('./data') if Path('./data').exists() else Path('./nuovo progetto')

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Rilevamento RAM di sistema
total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
logger.info(f"RAM di sistema rilevata: {total_ram_gb:.1f} GB")

# Configurazione Colab Premium:
# Con Colab Premium (High-RAM ~52 GB), usiamo PyArrow multi-threaded o mega-chunk
USE_COLAB_PREMIUM = total_ram_gb > 25.0 or True

if USE_COLAB_PREMIUM:
    logger.success("Modalità Google Colab Premium / High-RAM abilitata!")
    logger.info("Configurazione: Motore PyArrow multi-threaded attivo, mega-chunking (1.000.000 righe) e caricamento rapido in RAM.")
    DEFAULT_CHUNK_SIZE = 1_000_000
    DIRECT_LOAD_ALLOWED = True
else:
    logger.info("Modalità Colab Standard: streaming conservativo a 100.000 righe per blocco.")
    DEFAULT_CHUNK_SIZE = 100_000
    DIRECT_LOAD_ALLOWED = False

logger.info(f"Directory artifact di output: {DRIVE_DIR}")
logger.info(f"Directory dati grezzi selezionata: {RAW_DIR}")


Mounted at /content/drive
2026-09-19 08:13:41 | INFO     | RAM di sistema rilevata: 51.0 GB
2026-09-19 08:13:41 | SUCCESS  | Modalità Google Colab Premium / High-RAM abilitata!
2026-09-19 08:13:41 | INFO     | Configurazione: Motore PyArrow multi-threaded attivo, mega-chunking (1.000.000 righe) e caricamento rapido in RAM.
2026-09-19 08:13:41 | INFO     | Directory artifact di output: /content/drive/MyDrive/yelp_data
2026-09-19 08:13:41 | INFO     | Directory dati grezzi selezionata: /content/drive/MyDrive/data


## Rilevamento Automatico dei File Grezzi (JSON Lines o CSV)

In [4]:
def detect_yelp_sources(search_dirs):
    """
    Scansiona le directory alla ricerca dei file grezzi di Yelp,
    supportando sia il formato accademico JSON Lines (.json) sia i dump CSV Kaggle (.csv).
    Riconosce file come 'yelp_academic_dataset_business.json', ecc.
    """
    candidates = {}
    searched = []

    all_dirs = []
    for d in search_dirs:
        p = Path(d)
        if p.exists() and p not in all_dirs:
            all_dirs.append(p)

    # Controlla automaticamente cartelle in MyDrive contenenti 'data', 'yelp', 'raw'
    mydrive = Path('/content/drive/MyDrive')
    if mydrive.exists():
        if (mydrive / 'data').exists() and (mydrive / 'data') not in all_dirs:
            all_dirs.insert(0, mydrive / 'data')
        try:
            for sub in mydrive.glob('*'):
                if sub.is_dir() and any(k in sub.name.lower() for k in ['data', 'yelp', 'raw']):
                    if sub not in all_dirs:
                        all_dirs.append(sub)
        except Exception:
            pass

    for base in all_dirs:
        if not base.exists():
            continue
        searched.append(str(base))
        try:
            items = list(base.glob("*")) + list(base.glob("*/*"))
        except Exception:
            items = list(base.glob("*"))

        for f in items:
            if not f.is_file():
                continue
            name_lower = f.name.lower()
            if "business" in name_lower and ("json" in name_lower or "csv" in name_lower):
                candidates.setdefault("business", f)
            elif "review" in name_lower and ("json" in name_lower or "csv" in name_lower):
                candidates.setdefault("review", f)
            elif "user" in name_lower and ("json" in name_lower or "csv" in name_lower):
                candidates.setdefault("user", f)
            elif "checkin" in name_lower and ("json" in name_lower or "csv" in name_lower):
                candidates.setdefault("checkin", f)

    logger.info(f"Directory esaminate: {searched}")
    for key, path in candidates.items():
        size_mb = path.stat().st_size / (1024 * 1024)
        logger.success(f"Trovata sorgente '{key}': {path.name} ({size_mb:,.1f} MB) in {path.parent}")

    return candidates

# Cerca specificamente nella cartella 'data' di Drive, RAW_DIR, DRIVE_DIR e cartelle locali
possible_dirs = [
    Path('/content/drive/MyDrive/data'),
    RAW_DIR,
    DRIVE_DIR,
    Path('/content/drive/MyDrive'),
    Path('./data'),
    Path('./nuovo progetto'),
    Path('.')
]
source_files = detect_yelp_sources(possible_dirs)


2026-09-19 08:13:44 | INFO     | Directory esaminate: ['/content/drive/MyDrive/data', '/content/drive/MyDrive/yelp_data', '/content/drive/MyDrive', '.', '/content/drive/MyDrive/yelp_raw']
2026-09-19 08:13:44 | SUCCESS  | Trovata sorgente 'business': yelp_academic_dataset_business.json (113.4 MB) in /content/drive/MyDrive/data
2026-09-19 08:13:44 | SUCCESS  | Trovata sorgente 'checkin': yelp_academic_dataset_checkin.json (273.7 MB) in /content/drive/MyDrive/data
2026-09-19 08:13:44 | SUCCESS  | Trovata sorgente 'review': yelp_academic_dataset_review.json (5,094.4 MB) in /content/drive/MyDrive/data
2026-09-19 08:13:44 | SUCCESS  | Trovata sorgente 'user': yelp_academic_dataset_user.json (3,207.5 MB) in /content/drive/MyDrive/data


## Ingestion e Pulizia Business (`yelp_businesses.parquet`)

In [5]:
def process_businesses(source_path, output_parquet):
    start_t = time.perf_counter()
    logger.info(f"Inizio elaborazione dataset Business da {source_path}...")

    is_json = source_path.suffix.lower() == '.json'

    # Lettura rapida ad alte prestazioni
    if is_json:
        try:
            df = pd.read_json(source_path, lines=True, engine='pyarrow')
        except Exception:
            df = pd.read_json(source_path, lines=True)
    else:
        df = pd.read_csv(source_path, low_memory=False)

    initial_rows = len(df)
    logger.info(f"Business grezzi caricati: {initial_rows:,d} record")

    # Standardizzazione nomi colonne
    df.columns = df.columns.str.strip().str.lower()

    # 1. Filtro ristoranti e settore food
    cat_mask = df['categories'].fillna('').str.contains('restaurant|food|bars|cafes', case=False, regex=True)
    df = df[cat_mask].copy()

    # 2. Filtro business attivi (is_open == 1 se presente)
    if 'is_open' in df.columns:
        df = df[df['is_open'] == 1].copy()

    # 3. Selezione e normalizzazione campi
    key_cols = [
        'business_id', 'name', 'address', 'city', 'state', 'postal_code',
        'latitude', 'longitude', 'stars', 'review_count', 'is_open',
        'attributes', 'categories', 'hours'
    ]
    present_cols = [c for c in key_cols if c in df.columns]
    df = df[present_cols].copy()

    # Ottimizzazione tipi numerici
    df['stars'] = pd.to_numeric(df['stars'], errors='coerce').astype('float32')
    df['review_count'] = pd.to_numeric(df['review_count'], errors='coerce').fillna(0).astype('int32')
    if 'latitude' in df.columns and 'longitude' in df.columns:
        df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce').astype('float32')
        df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce').astype('float32')

    if hasattr(df, 'to_pandas'):
        df = df.to_pandas()

    # Salvataggio Parquet compresso
    df.to_parquet(output_parquet, index=False, compression='snappy')
    elapsed = time.perf_counter() - start_t
    size_mb = output_parquet.stat().st_size / (1024 * 1024)

    logger.success(f"Business filtrati salvati in: {output_parquet.name}")
    logger.info(f"Record finali: {len(df):,d} (da {initial_rows:,d}, riduzione {(1 - len(df)/initial_rows)*100:.1f}%) | Dimensione: {size_mb:.2f} MB | Tempo: {elapsed:.2f}s")
    return df

business_src = source_files.get('business')
if business_src and business_src.exists():
    df_businesses = process_businesses(business_src, DRIVE_DIR / 'yelp_businesses.parquet')
else:
    logger.warning("File sorgente Business non trovato nei percorsi monitorati. Creazione placeholder guidato...")
    demo_businesses = pd.DataFrame({
        'business_id': [f'b_{i:04d}' for i in range(1000)],
        'name': [f'Restaurant_{i}' for i in range(1000)],
        'address': [f'{i} Main St' for i in range(1000)],
        'city': np.random.choice(['Toronto', 'Las Vegas', 'Phoenix', 'Scottsdale'], size=1000),
        'state': np.random.choice(['ON', 'NV', 'AZ'], size=1000),
        'postal_code': ['M5V 2H1'] * 1000,
        'latitude': np.random.uniform(43.6, 43.8, size=1000).astype('float32'),
        'longitude': np.random.uniform(-79.5, -79.3, size=1000).astype('float32'),
        'stars': np.random.choice([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], size=1000).astype('float32'),
        'review_count': np.random.randint(5, 500, size=1000).astype('int32'),
        'is_open': [1] * 1000,
        'attributes': ["{'RestaurantsPriceRange2': '2', 'GoodForKids': 'True', 'BusinessParking': {'street': True}}"] * 1000,
        'categories': ['Restaurants, Italian, Pizza, Bars'] * 1000,
        'hours': ["{'Monday': '11:00-22:00'}"] * 1000
    })
    demo_businesses.to_parquet(DRIVE_DIR / 'yelp_businesses.parquet', index=False, compression='snappy')
    df_businesses = demo_businesses
    logger.info("Creato artifact yelp_businesses.parquet di riferimento.")


2026-09-19 08:13:44 | INFO     | Inizio elaborazione dataset Business da /content/drive/MyDrive/data/yelp_academic_dataset_business.json...


/usr/local/lib/python3.13/dist-packages/cudf/io/json.py:241: UserWarning: Using CPU via Pandas to read JSON dataset, this may be GPU accelerated in the future
  warnings.warn(


2026-09-19 08:13:51 | INFO     | Business grezzi caricati: 150,346 record
2026-09-19 08:13:56 | SUCCESS  | Business filtrati salvati in: yelp_businesses.parquet
2026-09-19 08:13:56 | INFO     | Record finali: 46,146 (da 150,346, riduzione 69.3%) | Dimensione: 4.68 MB | Tempo: 12.17s


## Ingestion Reviews con Supporto High-RAM / Mega-Chunk (`yelp_reviews.parquet`)

In [6]:
def process_reviews(source_path, target_business_ids, output_parquet, use_direct_load=DIRECT_LOAD_ALLOWED, chunksize=DEFAULT_CHUNK_SIZE):
    """
    Elabora le recensioni (file da 3-5 GB).
    Se use_direct_load è True (Colab Premium con ~52 GB RAM), esegue il caricamento diretto
    in memoria con motore multi-threaded PyArrow in circa 15-30 secondi!
    In alternativa, processa a blocchi di dimensione chunksize usando lo streaming Parquet nativo.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq

    start_t = time.perf_counter()
    is_json = source_path.suffix.lower() == '.json'
    target_set = set(target_business_ids)

    if output_parquet.exists():
        output_parquet.unlink()

    # OPZIONE 1: Caricamento diretto ad alte prestazioni (Colab Premium High-RAM)
    if use_direct_load and is_json:
        logger.info(f"Colab Premium High-RAM: Caricamento diretto ultra-rapido con PyArrow da {source_path.name}...")
        try:
            df = pd.read_json(source_path, lines=True, engine='pyarrow')
            ram_used = df.memory_usage().sum() / (1024**3)
            logger.info(f"Caricate {len(df):,d} recensioni grezze ({ram_used:.2f} GB RAM). Filtraggio su {len(target_set):,d} ristoranti...")
            df.columns = df.columns.str.strip().str.lower()

            # Filtro vettorizzato per business ristorativi
            mask = df['business_id'].isin(target_set)
            filtered = df[mask].copy()
            del df
            import gc; gc.collect()

            filtered['stars'] = pd.to_numeric(filtered['stars'], errors='coerce').astype('float32')
            for c in ['useful', 'funny', 'cool']:
                if c in filtered.columns:
                    filtered[c] = pd.to_numeric(filtered[c], errors='coerce').fillna(0).astype('int16')
            if 'date' in filtered.columns:
                filtered['date'] = pd.to_datetime(filtered['date'], errors='coerce')

            cols_to_keep = ['review_id', 'user_id', 'business_id', 'stars', 'date', 'text', 'useful', 'funny', 'cool']
            present = [c for c in cols_to_keep if c in filtered.columns]
            filtered = filtered[present]

            if hasattr(filtered, 'to_pandas'):
                filtered = filtered.to_pandas()

            filtered.to_parquet(output_parquet, index=False, compression='snappy')
            elapsed = time.perf_counter() - start_t
            size_mb = output_parquet.stat().st_size / (1024 * 1024)
            logger.success(f"Lettura diretta completata in {elapsed:.2f}s! Salvate {len(filtered):,d} recensioni ({size_mb:.2f} MB).")
            return len(filtered)
        except Exception as e:
            logger.warning(f"Caricamento diretto fallito o memoria insufficiente ({e}). Passaggio a streaming a blocchi...")

    # OPZIONE 2: Chunking streaming nativo con PyArrow ParquetWriter
    logger.info(f"Inizio elaborazione streaming Reviews con chunksize={chunksize:,d}...")

    total_processed = 0
    total_retained = 0
    writer = None

    if is_json:
        reader = pd.read_json(source_path, lines=True, chunksize=chunksize)
    else:
        reader = pd.read_csv(source_path, chunksize=chunksize, low_memory=False)

    for chunk_idx, chunk in enumerate(reader):
        total_processed += len(chunk)
        chunk.columns = chunk.columns.str.strip().str.lower()

        mask = chunk['business_id'].isin(target_set)
        filtered = chunk[mask].copy()

        if len(filtered) > 0:
            filtered['stars'] = pd.to_numeric(filtered['stars'], errors='coerce').astype('float32')
            for c in ['useful', 'funny', 'cool']:
                if c in filtered.columns:
                    filtered[c] = pd.to_numeric(filtered[c], errors='coerce').fillna(0).astype('int16')
            if 'date' in filtered.columns:
                filtered['date'] = pd.to_datetime(filtered['date'], errors='coerce')

            cols_to_keep = ['review_id', 'user_id', 'business_id', 'stars', 'date', 'text', 'useful', 'funny', 'cool']
            present = [c for c in cols_to_keep if c in filtered.columns]
            filtered = filtered[present]

            if hasattr(filtered, 'to_pandas'):
                filtered = filtered.to_pandas()

            table = pa.Table.from_pandas(filtered)
            if writer is None:
                writer = pq.ParquetWriter(str(output_parquet), table.schema, compression='snappy')
            writer.write_table(table)
            total_retained += len(filtered)

        logger.info(f"Blocco {chunk_idx + 1:2d} | Righe processate: {total_processed:,d} | Ritenute: {total_retained:,d}")

    if writer is not None:
        writer.close()

    elapsed = time.perf_counter() - start_t
    size_mb = output_parquet.stat().st_size / (1024 * 1024) if output_parquet.exists() else 0
    logger.success(f"Completata elaborazione Reviews: {total_retained:,d} recensioni salvate ({size_mb:.2f} MB, {elapsed:.2f}s)")
    return total_retained

review_src = source_files.get('review')
valid_bids = df_businesses['business_id'].unique()

if review_src and review_src.exists():
    process_reviews(review_src, valid_bids, DRIVE_DIR / 'yelp_reviews.parquet')
else:
    logger.warning("File sorgente Reviews non trovato. Generazione placeholder coerente...")
    n_rev = 25000
    demo_reviews = pd.DataFrame({
        'review_id': [f'r_{i:06d}' for i in range(n_rev)],
        'user_id': [f'u_{np.random.randint(0, 4000):05d}' for _ in range(n_rev)],
        'business_id': np.random.choice(valid_bids, size=n_rev),
        'stars': np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], p=[0.1, 0.1, 0.15, 0.35, 0.3], size=n_rev).astype('float32'),
        'date': pd.date_range('2020-01-01', periods=n_rev, freq='5min'),
        'text': np.random.choice([
            "Amazing food and exceptionally friendly staff! Loved the homemade pasta and cozy atmosphere.",
            "The steak was overcooked and the service was terribly slow. Had to wait 45 minutes for drinks.",
            "Great lunch spot with fresh ingredients. The pizza was crispy and delicious, will definitely return!",
            "Rude waiter, high prices, and small portions. Not worth the hype at all.",
            "Decent place for a quick bite. Nothing spectacular but clean and affordable."
        ], size=n_rev),
        'useful': np.random.randint(0, 10, size=n_rev).astype('int16'),
        'funny': np.random.randint(0, 5, size=n_rev).astype('int16'),
        'cool': np.random.randint(0, 5, size=n_rev).astype('int16')
    })
    demo_reviews.to_parquet(DRIVE_DIR / 'yelp_reviews.parquet', index=False, compression='snappy')
    logger.info(f"Creato yelp_reviews.parquet dimostrativo con {len(demo_reviews):,d} recensioni.")


2026-09-19 08:13:56 | INFO     | Colab Premium High-RAM: Caricamento diretto ultra-rapido con PyArrow da yelp_academic_dataset_review.json...


/usr/local/lib/python3.13/dist-packages/cudf/io/json.py:241: UserWarning: Using CPU via Pandas to read JSON dataset, this may be GPU accelerated in the future
  warnings.warn(


2026-09-19 08:16:44 | INFO     | Caricate 6,990,280 recensioni grezze (4.52 GB RAM). Filtraggio su 46,146 ristoranti...
2026-09-19 08:17:13 | SUCCESS  | Lettura diretta completata in 197.03s! Salvate 4,185,052 recensioni (1685.82 MB).


## Ingestion Utenti Attivi (`yelp_users.parquet`)

In [7]:
def process_users(source_path, active_user_ids, output_parquet, use_direct_load=DIRECT_LOAD_ALLOWED, chunksize=DEFAULT_CHUNK_SIZE):
    import pyarrow as pa
    import pyarrow.parquet as pq

    start_t = time.perf_counter()
    is_json = source_path.suffix.lower() == '.json'
    active_set = set(active_user_ids)

    if output_parquet.exists():
        output_parquet.unlink()

    # In Colab Premium, caricamento diretto rapido di Users (~1.5 GB)
    if use_direct_load and is_json:
        logger.info(f"Colab Premium: Lettura rapida in memoria dataset Users da {source_path.name}...")
        try:
            df = pd.read_json(source_path, lines=True, engine='pyarrow')
            df.columns = df.columns.str.strip().str.lower()
            filtered = df[df['user_id'].isin(active_set)].copy()
            del df
            import gc; gc.collect()

            cols_to_keep = ['user_id', 'name', 'review_count', 'yelping_since', 'friends', 'useful', 'funny', 'cool', 'fans', 'average_stars']
            present = [c for c in cols_to_keep if c in filtered.columns]
            filtered = filtered[present]

            if hasattr(filtered, 'to_pandas'):
                filtered = filtered.to_pandas()

            filtered.to_parquet(output_parquet, index=False, compression='snappy')
            elapsed = time.perf_counter() - start_t
            size_mb = output_parquet.stat().st_size / (1024 * 1024)
            logger.success(f"Utenti attivi salvati in {output_parquet.name}: {len(filtered):,d} record ({size_mb:.2f} MB, {elapsed:.2f}s)")
            return len(filtered)
        except Exception as e:
            logger.warning(f"Lettura diretta Users fallita ({e}). Passaggio a chunking...")

    # Modalità chunking streaming con PyArrow
    logger.info(f"Inizio elaborazione streaming Users con chunksize={chunksize:,d}...")
    total_retained = 0
    writer = None
    reader = pd.read_json(source_path, lines=True, chunksize=chunksize) if is_json else pd.read_csv(source_path, chunksize=chunksize, low_memory=False)

    for chunk in reader:
        chunk.columns = chunk.columns.str.strip().str.lower()
        filtered = chunk[chunk['user_id'].isin(active_set)].copy()
        if len(filtered) > 0:
            cols_to_keep = ['user_id', 'name', 'review_count', 'yelping_since', 'friends', 'useful', 'funny', 'cool', 'fans', 'average_stars']
            present = [c for c in cols_to_keep if c in filtered.columns]
            filtered = filtered[present]

            if hasattr(filtered, 'to_pandas'):
                filtered = filtered.to_pandas()

            table = pa.Table.from_pandas(filtered)
            if writer is None:
                writer = pq.ParquetWriter(str(output_parquet), table.schema, compression='snappy')
            writer.write_table(table)
            total_retained += len(filtered)

    if writer is not None:
        writer.close()

    elapsed = time.perf_counter() - start_t
    size_mb = output_parquet.stat().st_size / (1024 * 1024) if output_parquet.exists() else 0
    logger.success(f"Utenti attivi salvati in {output_parquet.name}: {total_retained:,d} record ({size_mb:.2f} MB, {elapsed:.2f}s)")
    return total_retained

user_src = source_files.get('user')
df_rev_temp = pd.read_parquet(DRIVE_DIR / 'yelp_reviews.parquet', columns=['user_id'])
active_uids = df_rev_temp['user_id'].unique()

if user_src and user_src.exists():
    process_users(user_src, active_uids, DRIVE_DIR / 'yelp_users.parquet')
else:
    logger.warning("File sorgente Users non trovato. Generazione placeholder coerente...")
    demo_users = pd.DataFrame({
        'user_id': active_uids,
        'name': [f'User_{i}' for i in range(len(active_uids))],
        'review_count': np.random.randint(1, 50, size=len(active_uids)).astype('int32'),
        'yelping_since': ['2018-05-12'] * len(active_uids),
        'friends': [','.join(np.random.choice(active_uids, size=np.random.randint(1, 6)).tolist()) for _ in range(len(active_uids))],
        'useful': np.random.randint(0, 100, size=len(active_uids)).astype('int32'),
        'average_stars': np.random.uniform(2.5, 4.8, size=len(active_uids)).astype('float32')
    })
    demo_users.to_parquet(DRIVE_DIR / 'yelp_users.parquet', index=False, compression='snappy')
    logger.info(f"Creato yelp_users.parquet dimostrativo con {len(demo_users):,d} utenti.")


2026-09-19 08:17:18 | INFO     | Colab Premium: Lettura rapida in memoria dataset Users da yelp_academic_dataset_user.json...


/usr/local/lib/python3.13/dist-packages/cudf/io/json.py:241: UserWarning: Using CPU via Pandas to read JSON dataset, this may be GPU accelerated in the future
  warnings.warn(


2026-09-19 08:19:21 | SUCCESS  | Utenti attivi salvati in yelp_users.parquet: 1,348,895 record (1835.39 MB, 123.29s)


## Ingestion Check-in (`yelp_checkins.parquet`)

In [8]:
def process_checkins(source_path, target_business_ids, output_parquet):
    start_t = time.perf_counter()
    logger.info(f"Inizio elaborazione Check-in da {source_path.name}...")

    is_json = source_path.suffix.lower() == '.json'
    target_set = set(target_business_ids)

    if is_json:
        try:
            df = pd.read_json(source_path, lines=True, engine='pyarrow')
        except Exception:
            df = pd.read_json(source_path, lines=True)
    else:
        df = pd.read_csv(source_path, low_memory=False)

    df.columns = df.columns.str.strip().str.lower()
    mask = df['business_id'].isin(target_set)
    filtered = df[mask].copy()

    if hasattr(filtered, 'to_pandas'):
        filtered = filtered.to_pandas()

    filtered.to_parquet(output_parquet, index=False, compression='snappy')
    elapsed = time.perf_counter() - start_t
    size_mb = output_parquet.stat().st_size / (1024 * 1024)
    logger.success(f"Check-in salvati in {output_parquet.name}: {len(filtered):,d} record ({size_mb:.2f} MB, {elapsed:.2f}s)")
    return filtered

checkin_src = source_files.get('checkin')
if checkin_src and checkin_src.exists():
    process_checkins(checkin_src, valid_bids, DRIVE_DIR / 'yelp_checkins.parquet')
else:
    logger.warning("File sorgente Checkin non trovato. Generazione placeholder coerente...")
    demo_checkins = pd.DataFrame({
        'business_id': valid_bids[:min(500, len(valid_bids))],
        'date': ['2020-01-01 12:00:00, 2020-01-02 18:30:00, 2020-01-03 20:00:00'] * min(500, len(valid_bids))
    })
    demo_checkins.to_parquet(DRIVE_DIR / 'yelp_checkins.parquet', index=False, compression='snappy')
    logger.info("Creato yelp_checkins.parquet dimostrativo.")


2026-09-19 08:19:21 | INFO     | Inizio elaborazione Check-in da yelp_academic_dataset_checkin.json...


/usr/local/lib/python3.13/dist-packages/cudf/io/json.py:241: UserWarning: Using CPU via Pandas to read JSON dataset, this may be GPU accelerated in the future
  warnings.warn(


2026-09-19 08:19:31 | SUCCESS  | Check-in salvati in yelp_checkins.parquet: 45,405 record (81.79 MB, 9.61s)


## Validazione Finale degli Artifact e Riepilogo Compressione

In [9]:
summary_rows = []
artifact_files = [
    'yelp_businesses.parquet',
    'yelp_reviews.parquet',
    'yelp_users.parquet',
    'yelp_checkins.parquet'
]

for name in artifact_files:
    p = DRIVE_DIR / name
    if not p.exists():
        raise FileNotFoundError(f"Artifact mancante: {p}")
    df_check = pd.read_parquet(p)
    size_mb = p.stat().st_size / (1024 * 1024)
    summary_rows.append({
        'Artifact': name,
        'Righe': f"{len(df_check):,d}",
        'Colonne': len(df_check.columns),
        'Dimensione (MB)': f"{size_mb:.2f}",
        'Campione Colonne': ", ".join(df_check.columns[:4])
    })

summary_df = pd.DataFrame(summary_rows)
logger.success("Fase 1 completata con successo in modalità Colab ad alte prestazioni!")
display(summary_df)

2026-09-19 08:19:33 | SUCCESS  | Fase 1 completata con successo in modalità Colab ad alte prestazioni!


,Artifact,Righe,Colonne,Dimensione (MB),Campione Colonne
0,yelp_businesses.parquet,"46,146",14,4.68,"business_id, name, address, city"
1,yelp_reviews.parquet,"4,185,052",9,1685.82,"review_id, user_id, business_id, stars"
2,yelp_users.parquet,"1,348,895",10,1835.39,"user_id, name, review_count, yelping_since"
3,yelp_checkins.parquet,"45,405",2,81.79,"business_id, date"
